# Phase H / v3.0 — G2: multi-step arithmetic depth sweep (standard trainable-attn base + full BP)
RESEARCH-ONLY, isolated. Sweeps n_steps to map how deep multi-step reasoning installs. Contrast: ZeroBP is chance at k=2 already (uninstallable at any BP depth). Real NL GSM8K = G2b stretch (generative; needs a causal-LM variant + scale) — not this kernel.

In [ ]:
%%writefile ph_base.py
"""Phase H / v3.0 base -- a STANDARD multi-layer trainable-attention Transformer (RESEARCH-ONLY).

Deliberately a clean break from the v1.0/v2.0 ZeroBP backbone (last-position collapse + single frozen
reservoir attention): this is a normal pre-LN Transformer with multi-head SELF-attention trained by full
backprop, and NON-collapsing readout (mean-pool over tokens). Pure torch, ZERO dependency on
`kaggle_zerograd_moe` -- this file (and the rest of `phase_h/`) can move to a separate repo unchanged.

Isolation (ADR-005): never imported by the submission path; never imports the submission module.
"""
import math
import torch
import torch.nn as nn


class PhConfig:
    def __init__(self, vocab, seq_len, d_model=128, n_layers=4, n_heads=4, d_ff=None, n_cls=3, dropout=0.1):
        self.vocab = vocab; self.seq_len = seq_len; self.d_model = d_model
        self.n_layers = n_layers; self.n_heads = n_heads; self.d_ff = d_ff or 4 * d_model
        self.n_cls = n_cls; self.dropout = dropout


class Block(nn.Module):                                            # pre-LN Transformer block (bidirectional self-attn + FFN)
    def __init__(self, c):
        super().__init__()
        self.ln1 = nn.LayerNorm(c.d_model); self.ln2 = nn.LayerNorm(c.d_model)
        self.attn = nn.MultiheadAttention(c.d_model, c.n_heads, dropout=c.dropout, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(c.d_model, c.d_ff), nn.GELU(),
                                nn.Dropout(c.dropout), nn.Linear(c.d_ff, c.d_model))
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x, key_padding_mask=None):
        a = self.ln1(x)
        a, _ = self.attn(a, a, a, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + self.drop(a)
        x = x + self.drop(self.ff(self.ln2(x)))
        return x


class PhTransformer(nn.Module):                                   # token+pos embedding -> N blocks -> mean-pool -> classifier
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab, c.d_model, padding_idx=0)
        self.pos = nn.Embedding(c.seq_len, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layers)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.n_cls)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x):                                          # x [B,T] long, pad token = 0
        B, T = x.shape
        pad = (x == 0)                                            # [B,T] True = pad (ignored by attn + pool)
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.drop(self.tok(x) + self.pos(pos))
        for blk in self.blocks:
            h = blk(h, key_padding_mask=pad)
        h = self.ln(h)
        m = (~pad).float().unsqueeze(-1)                          # non-collapsing readout: mean over non-pad tokens
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1.0)
        return self.head(pooled)

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


In [ ]:
%%writefile ph_gsm_gpu.py
"""Phase H / v3.0 -- G2: multi-step reasoning depth. RESEARCH-ONLY, isolated (no ZeroBP import).

Honest framing: the ZeroBP contrast on "multi-step" was CLASSIFICATION of k-step modular arithmetic
(ZeroBP: chance at k=2, at every BP depth -- uninstallable). The defensible, apples-to-apples G2 is to
scale that difficulty: does the standard trainable-attn base + full BP INSTALL multi-step reasoning, and
how far does it scale in the number of steps? We sweep n_steps and report accuracy per depth.

  (Real natural-language GSM8K = G2b STRETCH: it is generative + needs a causal-LM variant + pretraining/
   scale; a tiny from-scratch classifier does NOT do it. Not scaffolded here as a win -- see charter §2.3.)

Writes runs/ph_gsm_run_summary.json (per-depth acc + config) for orchestrator monitoring.
Run locally (smoke):  python3 phase_h/ph_gsm_gpu.py --steps_list 2,3,4 --train_steps 1500
On Kaggle T4:         python3 phase_h/ph_gsm_gpu.py --steps_list 2,4,6,8 --train_steps 6000 --d_model 256 --layers 6
"""
import os, sys, json, random, argparse, time
import torch
import torch.nn as nn
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from ph_base import PhConfig, PhTransformer

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def make_vocab(digit_range):                                      # PAD=0, digits 1..digit_range, then +/- ops
    PLUS, MINUS = digit_range + 1, digit_range + 2
    return PLUS, MINUS, digit_range + 3


def gen(n, seed, n_steps, mod, digit_range):
    PLUS, MINUS, _ = make_vocab(digit_range)
    rng = random.Random(seed); X, Y = [], []
    seq_len = 2 * n_steps + 1                                     # (n_steps+1) digits + n_steps ops
    for _ in range(n):
        ds = [rng.randrange(digit_range) for _ in range(n_steps + 1)]
        ops = [rng.choice([PLUS, MINUS]) for _ in range(n_steps)]
        toks, v = [ds[0] + 1], ds[0]
        for i in range(n_steps):
            v = v + ds[i+1] if ops[i] == PLUS else v - ds[i+1]
            toks += [ops[i], ds[i+1] + 1]
        X.append(toks); Y.append(v % mod)
    return X, Y, seq_len


@torch.no_grad()
def evaluate(model, X, Y, bs=512):
    model.eval(); c = 0
    for i in range(0, len(X), bs):
        c += int(model(X[i:i+bs].to(DEVICE)).argmax(-1).cpu().eq(Y[i:i+bs]).sum())
    return c / max(1, len(X))


def train_depth(n_steps, a):
    _, _, V = make_vocab(a.digit_range)
    Xtr, Ytr, seq = gen(a.n_train, a.seed+2, n_steps, a.mod, a.digit_range)
    Xv, Yv, _ = gen(a.n_val, a.seed+3, n_steps, a.mod, a.digit_range)
    Xtr_d = torch.tensor(Xtr, dtype=torch.long).to(DEVICE); Ytr_d = torch.tensor(Ytr, dtype=torch.long).to(DEVICE)
    Xv_t = torch.tensor(Xv, dtype=torch.long); Yv_t = torch.tensor(Yv, dtype=torch.long)
    cfg = PhConfig(vocab=V, seq_len=seq, d_model=a.d_model, n_layers=a.layers, n_heads=a.heads, n_cls=a.mod, dropout=0.1)
    model = PhTransformer(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=0.01); lossf = nn.CrossEntropyLoss()
    g = torch.Generator().manual_seed(a.seed)
    for step in range(1, a.train_steps+1):
        model.train(); ix = torch.randint(0, len(Xtr_d), (a.batch,), generator=g)
        loss = lossf(model(Xtr_d[ix]), Ytr_d[ix]); opt.zero_grad(); loss.backward(); opt.step()
    return evaluate(model, Xv_t, Yv_t), model.n_params()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--steps_list", default="2,3,4"); ap.add_argument("--mod", type=int, default=5)
    ap.add_argument("--digit_range", type=int, default=5)
    ap.add_argument("--layers", type=int, default=4); ap.add_argument("--heads", type=int, default=4)
    ap.add_argument("--d_model", type=int, default=128)
    ap.add_argument("--train_steps", type=int, default=1500); ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--lr", type=float, default=3e-4); ap.add_argument("--n_train", type=int, default=8000)
    ap.add_argument("--n_val", type=int, default=1500); ap.add_argument("--seed", type=int, default=1234)
    ap.add_argument("--out", default=os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "runs", "ph_gsm_run_summary.json"))
    a = ap.parse_args(); torch.manual_seed(a.seed); t0 = time.time()
    depths = [int(s) for s in a.steps_list.split(",")]
    chance = 100.0 / a.mod
    print(f"\n==== Phase H G2: multi-step arithmetic depth sweep  ({a.layers}L x {a.heads}H d={a.d_model}, "
          f"mod {a.mod}, chance {chance:.1f}%, dev={DEVICE.type}) ====")
    print(f"  {'n_steps':>8} {'val_acc':>9} {'params(M)':>10}")
    rows = []
    for k in depths:
        acc, nparams = train_depth(k, a)
        rows.append({"n_steps": k, "val_acc": round(acc, 4), "params_M": round(nparams/1e6, 3)})
        print(f"  {k:>8} {acc*100:>8.1f}% {nparams/1e6:>10.2f}")
    summ = {"phase": "H", "gate": "G2", "task": "multi-step-arithmetic", "mod": a.mod,
            "config": {"layers": a.layers, "heads": a.heads, "d_model": a.d_model, "train_steps": a.train_steps},
            "chance": round(chance/100, 4), "by_depth": rows,
            "zerobp_contrast": "ZeroBP: chance at k=2, every BP depth (uninstallable)",
            "wall_s": round(time.time()-t0, 1), "device": DEVICE.type}
    os.makedirs(os.path.dirname(a.out), exist_ok=True)
    with open(a.out, "w") as f: json.dump(summ, f, indent=2)
    best = max(rows, key=lambda r: r["n_steps"])
    print(f"\n  deepest k={best['n_steps']} -> {best['val_acc']*100:.1f}%  (ZeroBP: chance at k=2 already)")
    print(f"  summary -> {a.out}")
    return summ


if __name__ == "__main__":
    main()


In [ ]:
import sys, runpy
sys.argv = ['ph_gsm_gpu.py', '--steps_list', '2,4,6,8', '--train_steps', '6000', '--d_model', '256', '--layers', '6', '--heads', '8', '--out', '/kaggle/working/ph_gsm_run_summary.json']
runpy.run_path('ph_gsm_gpu.py', run_name='__main__')